# Agent 3 — Data Quality

**What this notebook does:**  
Audits the master dataset for missing values, outliers, and inconsistencies before any scoring begins. Flags data that needs manual attention and produces a quality report you can include in the appendix.

**How to present this to investors:**  
> *Before any scoring or portfolio construction, our data quality agent audits every field in the master dataset. Missing values are flagged rather than silently filled, outliers are documented, and a confidence tier (High / Medium / Low) is assigned to each data category. This audit trail satisfies due-diligence requirements and allows the panel to understand exactly where data limitations exist.*

**Run after:** `01_data_ingestion.ipynb`

## Step 1 — Load the master dataset

In [1]:
import pandas as pd
import numpy as np
import os
import json
from datetime import date
import glob

TODAY = str(date.today())

# Find the most recent master dataset
files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
if not files:
    raise FileNotFoundError("No master dataset found. Run 01_data_ingestion.ipynb first.")

master_path = files[-1]
master = pd.read_csv(master_path)
print(f"Loaded: {master_path}")
print(f"Shape: {master.shape[0]} companies x {master.shape[1]} columns")

Loaded: ../outputs/scores\master_dataset_2026-05-20.csv
Shape: 279 companies x 677 columns


## Step 2 — Missing-value audit by data category

We group columns by source (identifiers, ESG environmental, ESG governance, EU Taxonomy, market prices) and measure how much is missing in each group.

In [2]:
# === ACTION REQUIRED ===
# After running 01_data_ingestion, look at the column names in master
# and fill in representative column names for each group below.
# These are examples — update them to match your actual column names.

# Columns that identify the company (should be 100% complete)
ID_COLS = [c for c in master.columns if any(k in c.lower() for k in ["name", "ticker", "isin", "sector", "country", "bics"])]

# Columns from the ESG environmental/social file
ESG_COLS = [c for c in master.columns if any(k in c.lower() for k in ["ghg", "carbon", "emission", "scope", "water", "energy", "esg", "environmental", "social"])]

# Columns from the governance file
GOV_COLS = [c for c in master.columns if any(k in c.lower() for k in ["governance", "board", "diversity", "compensation", "audit"])]

# Columns from the EU Taxonomy file
TAX_COLS = [c for c in master.columns if any(k in c.lower() for k in ["taxonomy", "eligible", "alignment", "dnsh", "green"])]

def missing_report(df, cols, label):
    if not cols:
        print(f"\n{label}: no matching columns found — update the column lists above")
        return
    subset = df[cols]
    pct_missing = subset.isnull().mean().mul(100).round(1)
    n_complete = (pct_missing == 0).sum()
    n_partial  = ((pct_missing > 0) & (pct_missing < 100)).sum()
    n_empty    = (pct_missing == 100).sum()
    avg_miss   = pct_missing.mean().round(1)
    print(f"\n{'='*55}")
    print(f"  {label} ({len(cols)} columns)")
    print(f"{'='*55}")
    print(f"  Complete (0% missing):    {n_complete} columns")
    print(f"  Partial  (1-99% missing): {n_partial} columns")
    print(f"  Empty    (100% missing):  {n_empty} columns")
    print(f"  Average missing rate:     {avg_miss}%")
    if n_partial > 0:
        print(f"\n  Partially filled columns:")
        for col, pct in pct_missing[pct_missing > 0].sort_values(ascending=False).items():
            print(f"    {col:<45} {pct:>5.1f}% missing")

missing_report(master, ID_COLS,  "Identifiers")
missing_report(master, ESG_COLS, "ESG Environmental & Social")
missing_report(master, GOV_COLS, "Governance")
missing_report(master, TAX_COLS, "EU Taxonomy")


  Identifiers (12 columns)
  Complete (0% missing):    9 columns
  Partial  (1-99% missing): 3 columns
  Empty    (100% missing):  0 columns
  Average missing rate:     19.6%

  Partially filled columns:
    classificationLevelName7                       98.2% missing
    classificationLevelName6                       87.1% missing
    classificationLevelName5                       49.5% missing

  ESG Environmental & Social (209 columns)
  Complete (0% missing):    0 columns
  Partial  (1-99% missing): 205 columns
  Empty    (100% missing):  4 columns
  Average missing rate:     75.3%

  Partially filled columns:
    euTaxnmyEstmatdSubstantlContrbtnWaterAmountRevenue 100.0% missing
    pctPortfolioEnergyRtg                         100.0% missing
    cadmiumEmissions                              100.0% missing
    euTaxnmyEstmatdSubstantlContrbtnWaterPctRevenue 100.0% missing
    totGhgCo2EmIntensPerVehSld                     99.6% missing
    totGhgCo2EmIntensPerRpm                  

## Step 3 — Assign data confidence tiers

The Data Dictionary (required in the report appendix) classifies every variable by confidence.  
This cell auto-generates a draft classification based on missingness rates.

In [3]:
miss_pct = master.isnull().mean().mul(100).round(1)

def confidence_tier(pct):
    if pct == 0:     return "High"
    elif pct <= 20:  return "Medium"
    else:            return "Low"

data_dict_rows = []
for col in master.columns:
    pct = miss_pct[col]
    dtype = str(master[col].dtype)
    source = (
        "equityBicsV2" if col in ID_COLS else
        "esgEnvironmentalSocial" if col in ESG_COLS else
        "esgGovernance" if col in GOV_COLS else
        "legalEntityEuTaxonomy" if col in TAX_COLS else
        "yfinance / calculated"
    )
    data_dict_rows.append({
        "variable": col,
        "dtype": dtype,
        "source": source,
        "pct_missing": pct,
        "confidence": confidence_tier(pct),
        "data_type": "reported"   # update manually: reported/observed/estimated/AI-extracted/judgement-based
    })

data_dict = pd.DataFrame(data_dict_rows)
print("Data confidence summary:")
print(data_dict['confidence'].value_counts().to_string())
print("\nFirst 10 rows of data dictionary:")
data_dict.head(10)

Data confidence summary:
confidence
Low       498
Medium    148
High       31

First 10 rows of data dictionary:


,variable,dtype,source,pct_missing,confidence,data_type
0,rc,int64,yfinance / calculated,0.0,High,reported
1,idBbGlobal,str,yfinance / calculated,0.0,High,reported
2,idBbUnique,str,yfinance / calculated,0.0,High,reported
3,idBbCompany,int64,yfinance / calculated,0.0,High,reported
4,idBbGlobalCompany,str,yfinance / calculated,0.0,High,reported
5,idBbGlobalCompanyName,str,equityBicsV2,0.0,High,reported
6,idBbSecNumDes,str,yfinance / calculated,0.0,High,reported
7,ticker,str,equityBicsV2,0.0,High,reported
8,exchCode,str,yfinance / calculated,0.0,High,reported
9,idIsin,str,equityBicsV2,0.0,High,reported


## Step 4 — Outlier detection on key numeric fields

Outliers in emissions data are common (e.g. one company reports in kg while others use tonnes).  
This cell flags any value more than 3 standard deviations from the mean.

In [4]:
numeric_cols = master.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["data_vintage"]]

outlier_flags = []
for col in numeric_cols:
    series = master[col].dropna()
    if len(series) < 5:
        continue
    mean = series.mean()
    std  = series.std()
    if std == 0:
        continue
    z_scores = (series - mean) / std
    outliers = series[z_scores.abs() > 3]
    if len(outliers) > 0:
        # Find company names for the outlier rows
        name_col = next((c for c in ["companyName", "name", "company_name", "Name"] if c in master.columns), None)
        for idx in outliers.index:
            label = master.loc[idx, name_col] if name_col else f"Row {idx}"
            outlier_flags.append({
                "column": col,
                "company": label,
                "value": round(outliers[idx], 2),
                "mean": round(mean, 2),
                "z_score": round(z_scores[idx], 1),
                "action": "REVIEW — possible unit mismatch or data error"
            })

outliers_df = pd.DataFrame(outlier_flags)
if outliers_df.empty:
    print("No outliers found (z-score > 3). Data looks consistent.")
else:
    print(f"{len(outliers_df)} outlier values flagged for manual review:")
    print(outliers_df.to_string(index=False))

649 outlier values flagged for manual review:
                                            column company         value          mean  z_score                                        action
                                       idBbCompany Row 230  5.807608e+07  7.292782e+06      3.3 REVIEW — possible unit mismatch or data error
                                       idBbCompany Row 253  5.513037e+07  7.292782e+06      3.1 REVIEW — possible unit mismatch or data error
                                       idBbCompany Row 257  6.023874e+07  7.292782e+06      3.4 REVIEW — possible unit mismatch or data error
                                       idBbCompany Row 261  6.156424e+07  7.292782e+06      3.5 REVIEW — possible unit mismatch or data error
                                       idBbCompany Row 265  6.308789e+07  7.292782e+06      3.6 REVIEW — possible unit mismatch or data error
                                       idBbCompany Row 266  6.206820e+07  7.292782e+06      3.5 REVIEW

## Step 5 — EU Taxonomy note

This cell prints a reminder about the critical distinction between taxonomy **eligibility** and **alignment** — a common exam question.

In [5]:
print("=" * 65)
print("  EU TAXONOMY DATA QUALITY NOTE")
print("=" * 65)
print("""
IMPORTANT DISTINCTION:

  ELIGIBILITY  = the company's activities COULD qualify under the
                 EU Taxonomy. Data coverage is high (~80-90%).

  ALIGNMENT    = the company has REPORTED that activities meet
                 the Do No Significant Harm (DNSH) criteria AND
                 minimum social safeguards. Coverage is sparse
                 because reporting is voluntary until 2026.

  WARNING: Never present eligibility as alignment in the report
           or to the investor panel. They are NOT equivalent.

  For this portfolio we use ELIGIBILITY as a proxy for green
  revenue exposure, clearly labelled as an estimate.
""")

# Count how many companies have any alignment data
align_cols = [c for c in master.columns if "align" in c.lower()]
if align_cols:
    n_with_alignment = master[align_cols].notna().any(axis=1).sum()
    print(f"Companies with alignment data: {n_with_alignment} / {len(master)}")
    print(f"Companies with only eligibility: {len(master) - n_with_alignment} / {len(master)}")
else:
    print("(Alignment columns not detected — check column names after loading CSVs)")

  EU TAXONOMY DATA QUALITY NOTE

IMPORTANT DISTINCTION:

  ELIGIBILITY  = the company's activities COULD qualify under the
                 EU Taxonomy. Data coverage is high (~80-90%).

  ALIGNMENT    = the company has REPORTED that activities meet
                 the Do No Significant Harm (DNSH) criteria AND
                 minimum social safeguards. Coverage is sparse
                 because reporting is voluntary until 2026.

           or to the investor panel. They are NOT equivalent.

  For this portfolio we use ELIGIBILITY as a proxy for green
  revenue exposure, clearly labelled as an estimate.

Companies with alignment data: 158 / 279
Companies with only eligibility: 121 / 279


## Step 6 — Save quality report

In [6]:
os.makedirs("../outputs/scores", exist_ok=True)

# Save data dictionary (required in report appendix)
dict_path = f"../outputs/scores/data_dictionary_{TODAY}.csv"
data_dict.to_csv(dict_path, index=False)
print(f"Data dictionary saved: {dict_path}")

# Save outlier log
if not outliers_df.empty:
    outlier_path = f"../outputs/scores/outlier_flags_{TODAY}.csv"
    outliers_df.to_csv(outlier_path, index=False)
    print(f"Outlier log saved:     {outlier_path}")
else:
    print("No outliers to save.")

print("\nData quality check complete.")

Data dictionary saved: ../outputs/scores/data_dictionary_2026-05-20.csv
Outlier log saved:     ../outputs/scores/outlier_flags_2026-05-20.csv

Data quality check complete.


## ✅ Notebook complete

You now have:
- A **missing-value audit** by data category
- A **data dictionary** with confidence tiers (required in report appendix)
- An **outlier flag list** for manual review
- A written note on the eligibility vs. alignment distinction

**Next:** Open `02_financial_analysis.ipynb` to calculate returns, volatility, and Sharpe ratios.